# 1. Identifying Data Types

### Concept & Definition
Data type identification is the process of inspecting raw features to determine how each attribute is stored in memory (e.g., `object`, `int64`, `float64`, `datetime64`, `bool`). In Pandas, raw files often load numbers or dates as generic `object` (string) types due to missing values or symbol characters.

### Real-World / Business Example
In a customer database, `TotalCharges` might contain currency symbols (e.g., `"$1,200.50"`), causing Pandas to infer the column as an `object` (string) rather than a floating-point number.

### Why It Solves Problems & ML Impact
Machine Learning models execute matrix linear algebra. If a column is stored as an `object` instead of numeric, model trainers (like Scikit-Learn estimators) will throw immediate type errors during `fit()`.

In [1]:
import numpy as np
import pandas as pd

# Load dataset
df = pd.read_csv("Customer_Data.csv")

print("=== Raw Data Types Audit ===")
print(df.dtypes)

print("\n=== Data Overview & Memory Usage ===")
df.info()

=== Raw Data Types Audit ===
CustomerID            str
Age               float64
Gender                str
TenureYears         int64
MonthlyCharges        str
TotalCharges      float64
ContractType          str
PaymentMethod         str
Churn                 str
dtype: object

=== Data Overview & Memory Usage ===
<class 'pandas.DataFrame'>
RangeIndex: 1010 entries, 0 to 1009
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   CustomerID      1010 non-null   str    
 1   Age             956 non-null    float64
 2   Gender          962 non-null    str    
 3   TenureYears     1010 non-null   int64  
 4   MonthlyCharges  967 non-null    str    
 5   TotalCharges    1010 non-null   float64
 6   ContractType    1010 non-null   str    
 7   PaymentMethod   1010 non-null   str    
 8   Churn           1010 non-null   str    
dtypes: float64(2), int64(1), str(6)
memory usage: 114.3 KB


# 2. Numerical Data

### Concept & Definition
Numerical data represents continuous or discrete quantitative measurements (e.g., integers, 32-bit/64-bit floats). Continuous values can take any range (e.g., `MonthlyCharges`), while discrete values represent countable quantities (e.g., `TenureYears`).

### Real-World / Business Example
An e-commerce platform tracks `MonthlyCharges` as floats and `TenureYears` as discrete integers to calculate lifetime customer values.

### Why It Solves Problems & ML Impact
Numerical data can be processed directly by mathematical operations like gradient descent, variance calculation, and matrix multiplication.

In [2]:
# Inspecting existing numerical columns
num_cols = df.select_dtypes(include=["number"]).columns
print("Numerical Columns Identified:")
print(list(num_cols))

print("\nSummary Statistics of Numerical Features:")
display(df[num_cols].describe())

Numerical Columns Identified:
['Age', 'TenureYears', 'TotalCharges']

Summary Statistics of Numerical Features:


,Age,TenureYears,TotalCharges
count,956.000000,1010.000000,1010.000000
mean,43.044979,4.115842,3937.420581
std,22.560566,3.402536,2278.700617
min,-5.000000,0.000000,101.473573
25%,25.000000,1.000000,1977.387488
50%,45.000000,3.000000,3920.063402
75%,52.000000,8.000000,5897.808542
max,150.000000,10.000000,7981.220296


# 5. Categorical Data

### Concept & Definition
Categorical data represents qualitative variables divided into distinct labels or categories (e.g., Nominal variables with no order like `Gender`, or Ordinal variables with clear ranks like `ContractType`).

### Real-World / Business Example
Categorizing contract options into `"Month-to-month"`, `"One year"`, and `"Two year"`.

### Why It Solves Problems & ML Impact
Explicitly declaring categorical types enables memory optimization and allows specialized algorithms (like LightGBM or CatBoost) to natively process categories without explicit manual encoding.

In [3]:
# Inspecting object/categorical columns
cat_cols = df.select_dtypes(include=["object"]).columns
print("Categorical Columns Identified:")
print(list(cat_cols))

print("\nUnique Values per Categorical Column:")
for col in cat_cols:
    print(f"{col}: {df[col].nunique()} unique categories")

Categorical Columns Identified:
['CustomerID', 'Gender', 'MonthlyCharges', 'ContractType', 'PaymentMethod', 'Churn']

Unique Values per Categorical Column:
CustomerID: 1000 unique categories
Gender: 4 unique categories
MonthlyCharges: 5 unique categories
ContractType: 6 unique categories
PaymentMethod: 4 unique categories
Churn: 2 unique categories


C:\Users\abarn\AppData\Local\Temp\ipykernel_7640\808875895.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=["object"]).columns


# 4. Boolean Data

### Concept & Definition
Boolean data stores binary true/false state values (`True`/`False` or `1`/`0`).

### Real-World / Business Example
Tracking whether a customer has canceled their service: `Churn` expressed as `True` or `False`.

### Why It Solves Problems & ML Impact
Boolean attributes occupy minimal memory (1 byte per entry) and can be easily converted into binary integer indicators ($0$ or $1$) for binary classification targets.

In [4]:
# Creating and identifying Boolean feature flags
df_bool_demo = df.copy()
df_bool_demo["IsSenior"] = df_bool_demo["Age"].apply(
    lambda x: True if pd.notnull(x) and float(x) >= 60 else False
)

print("Boolean Feature Value Counts:")
print(df_bool_demo["IsSenior"].value_counts())
print(f"Data Type: {df_bool_demo['IsSenior'].dtype}")

Boolean Feature Value Counts:
IsSenior
False    841
True     169
Name: count, dtype: int64
Data Type: bool


# 5. Date/Time Data

### Concept & Definition
Date/Time data represents temporal timestamps (`datetime64[ns]`). Raw date values often import as plain strings.

### Real-World / Business Example
A column stored as `"2026-09-01"` must be recognized as a timestamp to extract analytical features like `SignupYear`, `SignupMonth`, `DayOfWeek`, or time elapsed.

### Why It Solves Problems & ML Impact
ML models cannot interpret date strings directly. Converting them to temporal types enables feature engineering based on time differences.

In [5]:
# Adding synthetic date string and converting to datetime64
df_time_demo = df.copy()
df_time_demo["SignupDate"] = "2025-01-15"

# Convert String -> Datetime
df_time_demo["SignupDate"] = pd.to_datetime(df_time_demo["SignupDate"])

print(f"SignupDate Data Type: {df_time_demo['SignupDate'].dtype}")
print("\nExtracted Features:")
print("Year:", df_time_demo["SignupDate"].dt.year.iloc[0])
print("Month:", df_time_demo["SignupDate"].dt.month.iloc[0])
print("Day Name:", df_time_demo["SignupDate"].dt.day_name().iloc[0])

SignupDate Data Type: datetime64[us]

Extracted Features:
Year: 2025
Month: 1
Day Name: Wednesday


# 6. String Data

### Concept & Definition
String data encompasses unstructured or free-form text attributes, identifiers, or uncleaned entries stored under the generic `object` dtype in Pandas.

### Real-World / Business Example
User IDs (e.g., `"CUST-1001"`), physical addresses, or user comments.

### Why It Solves Problems & ML Impact
Identifies high-cardinality metadata (like primary keys) that should be dropped before modeling, preventing artificial memory overhead and noise.

In [6]:
# Identifying string metadata features
string_cols = [
    col
    for col in df.columns
    if df[col].dtype == "object" and df[col].nunique() == len(df)
]
print(f"Unique String Identifier Columns to Exclude from Modeling: {string_cols}")

Unique String Identifier Columns to Exclude from Modeling: []


# 7. Converting Data Types using astype()

### Concept & Definition
The `astype()` method explicitly casts a Pandas Series from one data type to another (e.g., casting an integer to a float, or an object string to a categorical type).

### Real-World / Business Example
Converting `Gender` from a generic object string into a Pandas `category` type to optimize memory usage.

### Why It Solves Problems & ML Impact
Reduces overall RAM consumption and enforces strict type constraints across processing steps.

In [7]:
# Memory footprint comparison before and after astype()
mem_before = df["Gender"].memory_usage(deep=True)

df_casted = df.copy()
df_casted["Gender"] = df_casted["Gender"].astype("category")

mem_after = df_casted["Gender"].memory_usage(deep=True)

print(f"Memory Usage Before (object): {mem_before} bytes")
print(f"Memory Usage After (category): {mem_after} bytes")
print(
    f"Memory Reduction: {((mem_before - mem_after) / mem_before) * 100:.2f}%"
)

Memory Usage Before (object): 12542 bytes
Memory Usage After (category): 1187 bytes
Memory Reduction: 90.54%


# 8. Converting Data Types using to_numeric()

### Concept & Definition
`pd.to_numeric()` safely parses numerical values embedded inside string columns, providing error-handling parameters (like `errors='coerce'`) to manage invalid text gracefully.

### Real-World / Business Example
Parsing `MonthlyCharges` containing values like `"$29.85"`, `"99.99"`, and corrupt strings like `"INVALID"`.

### Why It Solves Problems & ML Impact
Unlike standard `astype()`, `to_numeric()` handles invalid non-numeric strings by converting them to `NaN` instead of crashing your script.

In [8]:
# Safe conversion using pd.to_numeric
df_num_demo = df.copy()

# Clean currency signs first, then coerce errors to NaN
cleaned_string_series = (
    df_num_demo["MonthlyCharges"].astype(str).str.replace("$", "")
)
df_num_demo["MonthlyCharges_Clean"] = pd.to_numeric(
    cleaned_string_series, errors="coerce"
)

print("Original Values vs Cleaned Numeric Values:")
display(
    df_num_demo[["MonthlyCharges", "MonthlyCharges_Clean"]].head(10)
)
print(f"New Data Type: {df_num_demo['MonthlyCharges_Clean'].dtype}")

Original Values vs Cleaned Numeric Values:


,MonthlyCharges,MonthlyCharges_Clean
0,$29.85,29.85
1,$56.95,56.95
2,$105.50,105.50
3,$56.95,56.95
4,$29.85,29.85
5,$29.85,29.85
6,$29.85,29.85
7,$56.95,56.95
8,99.99,99.99
9,$29.85,29.85


New Data Type: float64


# 9. Converting Data Types using to_datetime()

### Concept & Definition
`pd.to_datetime()` standardizes heterogeneous string dates into unified `datetime64[ns]` timestamps.

### Real-World / Business Example
Parsing mixed date strings such as `"2025/01/15"`, `"15-Jan-2025"`, and `"2025-01-15T00:00:00"`.

### Why It Solves Problems & ML Impact
Converts human-readable date strings into standard UNIX timestamp representations that can be used to extract time-elapsed numeric features.

In [9]:
# Parsing heterogeneous date formats safely
date_strings = pd.Series(["2025-01-01", "2025/02/15", "invalid_date"])
parsed_dates = pd.to_datetime(date_strings, errors="coerce")

print("Parsed Datetime Series:")
print(parsed_dates)

Parsed Datetime Series:
0   2025-01-01
1          NaT
2          NaT
dtype: datetime64[us]


# 10. Handling Invalid Conversions

### Concept & Definition
Invalid conversions occur when non-standard values (e.g., `"N/A"`, `"null"`, `"INVALID"`, `"-"`) break standard numerical or date conversion operations.

### Parameter Strategies:
- `errors='raise'`: Raises an exception and halts execution (default behavior).
- `errors='ignore'`: Returns original un-converted input.
- `errors='coerce'`: Converts all invalid parsing attempts to `NaN` (preferred for downstream ML imputation pipelines).

### Real-World / Business Example
Cleaning legacy database records containing string values like `"UNKNOWN"` or `"N/A"` inside a numeric credit score column.

### Why It Solves Problems & ML Impact
Using `errors='coerce'` isolates corrupted entries into structured `NaN` markers, allowing them to be handled by numerical imputers in later steps.

In [10]:
# Demonstrating handling options for invalid data
dirty_series = pd.Series(["100", "200", "INVALID_TEXT", "400"])

print("--- Handled with errors='coerce' (Recommended for ML) ---")
coerced_series = pd.to_numeric(dirty_series, errors="coerce")
print(coerced_series)
print(f"Null Count Created: {coerced_series.isnull().sum()}")

--- Handled with errors='coerce' (Recommended for ML) ---
0    100.0
1    200.0
2      NaN
3    400.0
dtype: float64
Null Count Created: 1


# 11. Detecting Incorrect Data Types

### Concept & Definition
Detecting incorrect data types involves auditing features where stored data types do not align with their actual domain contents (e.g., numeric features stored as strings, or categorical target flags stored as floats).

### Real-World / Business Example
Detecting that `TotalCharges` is stored as an `object` because of whitespace or empty string characters.

### Why It Solves Problems & ML Impact
Automates data validation, preventing improper feature transformations during downstream steps.

In [11]:
# Automated audit function to detect hidden numerical features stored as objects
def audit_incorrect_dtypes(dataframe):
    incorrect_cols = []
    for col in dataframe.select_dtypes(include=["object"]).columns:
        # Check if stripping spaces and parsing allows numeric conversion
        coerced = pd.to_numeric(
            dataframe[col].astype(str).str.replace("$", ""), errors="coerce"
        )
        non_null_ratio = coerced.notnull().sum() / len(dataframe)
        if non_null_ratio > 0.5 and dataframe[col].nunique() > 10:
            incorrect_cols.append(col)
    return incorrect_cols


detected_cols = audit_incorrect_dtypes(df)
print(f"Columns stored as 'object' but containing numeric values: {detected_cols}")

Columns stored as 'object' but containing numeric values: []


C:\Users\abarn\AppData\Local\Temp\ipykernel_7640\1337536160.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in dataframe.select_dtypes(include=["object"]).columns:


# 12. Type Conversion Examples

### Overview of Essential Conversions
1. **String → Numeric:** Converting formatted charge strings into continuous float representations.
2. **String → Date:** Standardizing text timestamps into datetime objects.
3. **Numeric → Categorical:** Binning continuous variables or converting encoded numerical flags (e.g., `0`, `1`) into qualitative categories.
4. **Integer → Float:** Adjusting numerical precision to accommodate `NaN` missing values.

### Why It Solves Problems & ML Impact
Ensures each column matches the specific data type expected by transformers in Scikit-Learn pipelines.

In [12]:
df_conv = df.copy()

# 1. String -> Numeric
df_conv["MonthlyCharges_Num"] = pd.to_numeric(
    df_conv["MonthlyCharges"].astype(str).str.replace("$", ""), errors="coerce"
)

# 2. String -> Date
df_conv["CreatedDate"] = pd.to_datetime("2026-01-01")

# 3. Numeric -> Categorical
df_conv["Tenure_Cat"] = pd.cut(
    df_conv["TenureYears"],
    bins=[-1, 2, 5, 10],
    labels=["Short-Term", "Medium-Term", "Long-Term"],
)

# 4. Integer -> Float
df_conv["Tenure_Float"] = df_conv["TenureYears"].astype(float)

print("=== Converted DataFrame Dtypes ===")
print(df_conv[["MonthlyCharges_Num", "CreatedDate", "Tenure_Cat", "Tenure_Float"]].dtypes)
print("\nSample Output:")
display(df_conv[["MonthlyCharges_Num", "CreatedDate", "Tenure_Cat", "Tenure_Float"]].head(5))

=== Converted DataFrame Dtypes ===
MonthlyCharges_Num           float64
CreatedDate           datetime64[us]
Tenure_Cat                  category
Tenure_Float                 float64
dtype: object

Sample Output:


,MonthlyCharges_Num,CreatedDate,Tenure_Cat,Tenure_Float
0,29.85,2026-01-01,Medium-Term,3.0
1,56.95,2026-01-01,Long-Term,10.0
2,105.50,2026-01-01,Short-Term,2.0
3,56.95,2026-01-01,Long-Term,10.0
4,29.85,2026-01-01,Medium-Term,3.0


# 13. Summary: Why Correct Data Types are Important

### Core Takeaways
1. **Memory Efficiency:** Standard object dtypes consume significantly more RAM than proper categorical, boolean, or compact integer/float dtypes.
2. **Mathematical Correctness:** Machine Learning models require strict numerical inputs to perform matrix math.
3. **Pipeline Automation:** Correct dtypes allow `ColumnTransformer` to route numerical and categorical columns to their respective preprocessing pipelines automatically.
4. **Data Integrity:** Type validation prevents corrupt or invalid strings from sneaking into model training loops.

In [13]:
# Save clean data type version for subsequent sprints/notebooks
df_clean_dtypes = df.copy()

# Convert MonthlyCharges to float
df_clean_dtypes["MonthlyCharges"] = pd.to_numeric(
    df_clean_dtypes["MonthlyCharges"].astype(str).str.replace("$", ""),
    errors="coerce",
)

# Convert Age to numeric float (to support NaNs)
df_clean_dtypes["Age"] = pd.to_numeric(df_clean_dtypes["Age"], errors="coerce")

print("=== Final Validated Data Types ===")
print(df_clean_dtypes.dtypes)
print("\nNotebook 02 execution completed successfully!")

=== Final Validated Data Types ===
CustomerID            str
Age               float64
Gender                str
TenureYears         int64
MonthlyCharges    float64
TotalCharges      float64
ContractType          str
PaymentMethod         str
Churn                 str
dtype: object

Notebook 02 execution completed successfully!
